# CreteValley CIM data import and DPsim powerflow simulation 

## Import Libraries

In [3]:
import glob
import sys
import dpsim
import logging
import cimpy as cimpy
from villas.dataprocessing.readtools import *

## Import CIM data

In [4]:
# Remove all existing handlers to prevent multiple logging configurations
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(filename='CIMimport.log', level=logging.INFO, filemode='w')

# filename = './Network_CIM_Data/Crete_equivalent_min_loading_activeOnly/'
filename = '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/'
files = glob.glob(filename + '*.xml')
print(files)
import_result = cimpy.cim_import(files, "cgmes_v2_4_15")

with open('CIMimport.log', 'r') as file:
    for line in file:
        print(line.strip())

['/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___DL_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___GL_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SSH_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SV_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___TP_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z__EQ_.xml']
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___DL_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___GL_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SSH_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SV_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/

In [6]:
name = 'cv_me_1'
reader = dpsim.CIMReader(name)
system = reader.loadCIM(50, files, dpsim.Domain.SP, dpsim.PhaseType.Single, dpsim.GeneratorType.PVNode)

for comp in system.components:
    # Only components that are generators
    if comp.__class__.__name__ == "GeneratingUnit":
        # Access the setPointVoltage method
        v_set = comp.setPointVoltage()
        if v_set == 0:
            comp.setPointVoltage(1.0)  # set default 1.0 pu
            print(f"Set voltage for {comp.name()} to 1.0 pu")


CIMContentHandler: Note: 0 out of 5524 tasks remain unresolved!


[10:11:51.090253 lne_90135_90136_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:11:51.092295 lne_90131_90134_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
Uninitalized setPointVoltage for GeneratingUnit genstat_290531_W3. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_290131_W3. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_202101_PO. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_290231_W3. Using default value of 0
[10:11:51.098560 lne_61635_61933_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:11:51.098703 lne_61635_61735_2 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:11:51.101280 lne_90131_90133_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:11:51.102239 lne_61735_61833_1 warning] Zero value for Capacitance, setting default

## Run DPsim simulation

In [ ]:
sim = dpsim.Simulation(name, dpsim.LogLevel.info)
#logger = dpsim.Logger(name)

#for node in system.nodes:
#    logger.log_attribute(node.name()+'.V', 'v', node)
#    logger.log_attribute(node.name()+ '.S', 's', node)

#for comp in system.components:
    #if comp.__class__.__name__ =='PiLine':
      #  logger.log_attribute(comp.name() + '.Ibranch', 'current_vector', comp)
      #  logger.log_attribute(comp.name() + '.Pbranch', 'p_branch_vector', comp)
     #   logger.log_attribute(comp.name() + '.Qbranch', 'q_branch_vector', comp)
   # if comp.__class__.__name__ =='Transformer':
   #     logger.log_attribute(comp.name() + '.Ibranch', 'current_vector', comp)
   #     logger.log_attribute(comp.name() + '.Pbranch', 'p_branch_vector', comp)
   #     logger.log_attribute(comp.name() + '.Qbranch', 'q_branch_vector', comp)

                     
sim.set_system(system)
sim.set_time_step(1)
sim.set_final_time(2)
sim.set_domain(dpsim.Domain.SP)
sim.set_solver(dpsim.Solver.NRP)
#sim.set_solver(dpsim.Solver.SP)
sim.set_solver_component_behaviour(dpsim.SolverBehaviour.Simulation)
#sim.add_logger(logger)
sim.run()

## Read DPsim results

In [1]:
dpsim_result_file = 'logs/' + name + '.csv'
ts_dpsim = read_timeseries_csv(dpsim_result_file)

NameError: name 'name' is not defined

In [ ]:
import math, cmath

dpsim_bus_results = []

for node in system.nodes:
    row= {'name': node.name(),
          'vm [kV]': cmath.polar(ts_dpsim[node.name()+'.V'].values[0])[0]/1e3,
          'va [deg]': math.degrees(cmath.polar(ts_dpsim[node.name()+'.V'].values[0])[1]),
          'p_inj [MW]':ts_dpsim[node.name()+'.S'].values[0].real/1e6,
          'q_inj [MVAR]':ts_dpsim[node.name()+'.S'].values[0].imag/1e6}
    dpsim_bus_results.append(row)

        

df_dpsim_bus_results = pd.DataFrame(dpsim_bus_results, columns=['name', 'vm [kV]', 'va [deg]', 'p_inj [MW]', 'q_inj [MVAR]'])
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.3f}'.format
df_dpsim_bus_results

In [ ]:
import math, cmath

dpsim_branch_results = []

for comp in system.components:
    if comp.__class__.__name__ =='PiLine' or comp.__class__.__name__ =='Transformer':
        row= {'name': comp.name(),
              'i_0 [kA]': abs(ts_dpsim[comp.name()+'.Ibranch_0'].values[0]/1e3),  
              'p_0 [MW]': ts_dpsim[comp.name()+'.Pbranch_0'].values[0]/1e6,
              'q_0 [MVAR]': ts_dpsim[comp.name()+'.Qbranch_0'].values[0]/1e6,
              'i_1 [kA]': abs(ts_dpsim[comp.name()+'.Ibranch_1'].values[0]/1e3),
              'p_1 [MW]': ts_dpsim[comp.name()+'.Pbranch_1'].values[0]/1e6,
              'q_1 [MVAR]': ts_dpsim[comp.name()+'.Qbranch_1'].values[0]/1e6}
        dpsim_branch_results.append(row)

df_dpsim_branch_results = pd.DataFrame(dpsim_branch_results, columns=['name', 'i_0 [kA]', 'p_0 [MW]', 'q_0 [MVAR]', 'i_1 [kA]', 'p_1 [MW]', 'q_1 [MVAR]'])
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.3f}'.format

In [ ]:
with pd.ExcelWriter('test_scenario_dpsim_results.xlsx') as writer:
    df_dpsim_summary_results.to_excel(writer, sheet_name='summary', index=False)
    df_dpsim_bus_results.to_excel(writer, sheet_name='busses', index=False)
    df_dpsim_branch_results.to_excel(writer, sheet_name='branches', index=False)